# Gemma A/B-constrained protocol v1 — 18 October 2026

Experiment: 50 records, all four variants and both mappings, plus eight authored clean controls. Native and constrained arms. 928 judgments + 8 identity calls. Formatting and classification quality are separate outcomes. Base PR #3 / 03b32cb. No original pilot files are changed.


In [ ]:
import subprocess, sys, json, hashlib, zipfile, shutil
from pathlib import Path
from google.colab import files
BASE = "03b32cb2ac6ae6aabf67a6b2da7632c3de852834"
REPO = Path('/content/algoverse-ab-v1-code')
RUN = Path('/content/gemma-ab-v1-2026-10-18')
subprocess.run(['git','clone','https://github.com/Dharshan2004/algoverse.git',str(REPO)],check=True)
subprocess.run(['git','checkout','--detach',BASE],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==BASE
subprocess.run([sys.executable,'-m','pip','install','transformers==5.19.0','accelerate==1.15.0','bitsandbytes==0.50.2','huggingface_hub==1.33.0','mistral-common==1.12.0','sentencepiece','numpy'],check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
print('GPU:',torch.cuda.get_device_name(0),'torch:',torch.__version__,'base:',BASE)


## Upload private input and versioned overlay
Upload `ab-v1-private-input.zip`. It contains the original candidate JSONL and reviewed additive code. It is kept out of Git. The overlay hashes below freeze what is executed.


In [ ]:
uploaded=files.upload()
assert len(uploaded)==1
archive=Path('/content/ab-v1-private-input.zip');archive.write_bytes(next(iter(uploaded.values())))
DATA=Path('/content/ab-v1-input');DATA.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert all(not n.startswith('/') and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(DATA)
EXPECTED = {'inference.py': '7b3c4e0acd60ac12342e3652aaebaffa7a3df4ea181750bd46d87863c8b7aefd', 'ab_constraint.py': '95b4886d5b5c8c1dedf1892d95d8a5de0b47a0eccc70d573e0d0949ab9290b91', 'ab_experiment.py': 'b6f8eeb14a7b07bb28912514b6d3f309b13219993de10ef5fea1bafac5dd555a', 'ab_report.py': '3a47868cb7569d5ea59a93c5b5261c5ecfdc89ae5b8450fe887fba0b6294561d', 'pilot_report.py': '079afad379fcf33d49f6645d7d2b493a1baf3278d1ec63dc63feedf6d41ab990', 'controls/ab_v1.jsonl': '2b0475ffd1d2502e4744b8b7676c220b2b329353c151bb6883d83854ac0c61dc', 'AB_PROTOCOL.md': '33cb6b6c9bee749c0b2253cb80814c326183d3e284799eef33295d183b387cde'}
for name,digest in EXPECTED.items():
    p=DATA/'overlay'/name
    assert hashlib.sha256(p.read_bytes()).hexdigest()==digest,name
    dest=REPO/name;dest.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dest)
assert hashlib.sha256((DATA/'candidate_inputs.jsonl').read_bytes()).hexdigest()=='ae4f5d179571f290910c688ea59e68dae3110a74eebadd1810bfcaf91deda311'
print('Pinned base, input and overlay verified. Ready for 936 calls.')


In [ ]:
with Path('/content/ab-v1-execution.log').open('w') as log:
    command=[sys.executable,'-u',str(REPO/'ab_experiment.py'),'--input',str(DATA/'candidate_inputs.jsonl'),'--out',str(RUN)]
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout:
            log.write(line);log.flush();print(line,end='',flush=True)
        assert proc.wait()==0,'Inference failed; preserve partial outputs'
shutil.copy2('/content/ab-v1-execution.log',RUN/'execution.log')
subprocess.run([sys.executable,str(REPO/'ab_report.py'),str(RUN),'--out',str(RUN/'audit')],check=True)
print('AUDITED: see aggregate summary in the next cell.')


In [ ]:
summary=json.loads((RUN/'audit/summary.json').read_text())
for name,arm in summary['arms'].items():
    q=arm['quality']
    print(name,': valid',q['format_valid_count'],'/',q['rows'],'; source-label accuracy',round(q['accuracy_all_pct'],2),'%')
print(json.dumps(summary['comparisons'],indent=2))
files.download(shutil.make_archive(str(RUN),'zip',RUN))
